# ECONOMETRIA I (Repaso con ejercicio y Testeo de supuestos)

## Modelos de Regresión simple y múltiple

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.stats.api as sms
import matplotlib.pyplot as plt
import seaborn as sns

### Bibliografia

Gujarati, D., & Porter, D. (2010). Econometría (5ta. edición ed.). México DF: Mc Graw Hill. Parte 1

#### Modelos reciprocos
La tabla 6.4 presenta datos transversales de 64 países sobre mortalidad infantil y otras variables

In [ ]:
#. Cargamos dataset de Gujarati con la tabla 6.4

data = pd.read_csv('Tabla 6_4.csv')

In [ ]:
data

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.mi.describe()

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(data.pbi_pc , data.mi)
plt.xlabel('pbi per capita')
plt.ylabel('mortalidad infantil')
plt.title('Mortalidad Infantil vs PBIpc',fontsize=20)


In [ ]:
model_1 = sm.OLS(data.mi, data.pbi_pc)

In [ ]:
result_1 = model_1.fit()

In [ ]:
print(result_1.summary())

In [ ]:
model_2 = sm.OLS(data.mi, sm.add_constant(data.pbi_pc))

In [ ]:
result_2 = model_2.fit()

In [ ]:
print(result_2.summary())

In [ ]:
fig, ax=plt.subplots(figsize=(10,5))
ax.scatter(data.pbi_pc, data.mi, label='datos tabla',color='blue')
ax.scatter(data.pbi_pc, result_2.predict(), label = 'datos estimados' , color ='red')
ax.legend()
plt.show()
plt.savefig('graf1')

In [ ]:
#Ajustamos la variable para capturar adecuadamente e efecto. Los cambios en la variable transformada son más grandes cuando pbi_pc es bajo y más pequeños cuando pbi_pc es alto

data['1/pbi_pc'] = 1 / data.pbi_pc

In [ ]:
data

In [ ]:
model_3 = sm.OLS(data.mi, sm.add_constant(data['1/pbi_pc']))

In [ ]:
result_3 = model_3.fit()

In [ ]:
print(result_3.summary2())

In [ ]:
fig, ax=plt.subplots(figsize=(10,5))
ax.scatter(data.pbi_pc, data.mi, label='datos tabla',color='blue')
ax.scatter(data.pbi_pc, result_3.predict(), label = 'datos estimados' , color ='red')

plt.show()

In [ ]:
Y = data['mi']
X = data[['pbi_pc','taf']]

model_1 = sm.OLS(Y, sm.add_constant(X))
result_1 = model_1.fit()

In [ ]:
print(result_1.summary())

In [ ]:
result_1.params

In [ ]:
result_1.ssr

In [ ]:
residuos = result_1.resid

#### Modelos logaritmicos
La tabla 6.3 presenta datos sobre el gasto de consumo personal total (GCPERT), el gasto en
bienes duraderos (GASBD), el gasto en bienes perecederos (GASBPER) y el gasto en servicios
(GASERV), todos medidos en miles de millones de dólares de 2000

In [ ]:
df = pd.read_excel("Tabla 6_3.xlsx")


df.head()



In [ ]:
# Transformacion de variables

df["ln_GASBD"] = np.log(df.GASBD)
df["ln_GCPERT"] = np.log(df.GCPERT)

df

In [ ]:
X = df[['ln_GCPERT']]

modelo = sm.OLS(df.ln_GASBD, sm.add_constant(X))

resultado = modelo.fit()

print(resultado.summary2())

#### Añadir Dummys

In [ ]:
df["D_2006"] = (df["Año o trimestre"].str[:4] == "2006").astype(int)
df["D_Q4"] = (df["Año o trimestre"].str[-1] == "V").astype(int)
df

In [ ]:
X = df[['ln_GCPERT', "D_2006", "D_Q4"]]

modelo = sm.OLS(df.GASBD, sm.add_constant(X))

resultado = modelo.fit()

print(resultado.summary2())

## Test de supuestos

1. Linealidad
El modelo debe ser lineal en los parámetros. Esto significa que la relación entre la variable dependiente y las variables independientes es una función lineal de los parámetros.


2. Media condicional cero del término de error. E(u | X) = 0
La esperanza del término de error dado cualquier nivel de la variable independiente debe ser cero.
Esto garantiza que los estimadores no estén sesgados y tengan varianza mínima.


3. Homocedasticidad. Var(u | X) = σ^2
La varianza del término de error es constante para todos los niveles de la variable independiente. Esto es, los errores tienen la misma varianza sin importar el valor de X.


4. No autocorrelación de los errores. Cov(u_i, u_j | X) = 0 para todo i ≠ j
Los términos de error son independientes entre sí, es decir, el término de error en una observación no está correlacionado con el término de error en otra observación.

5. No colinealidad perfecta. Rango(X) = k (donde k es el número de parámetros estimados)
No debe haber relaciones lineales perfectas entre las variables independientes.
Esto asegura que la matriz X tenga rango completo y, por lo tanto, (X'X) sea invertible.

6. : Normalidad de los errores. u ~ N(0, σ^2)
Para inferencia sobre los estimadores, como pruebas de hipótesis y construcción de intervalos de confianza, se asume que los errores siguen una distribución normal.



### Bibliografia

Gujarati, D., & Porter, D. (2010). Econometría (5ta. edición ed.). México DF: Mc Graw Hill. Parte 2

In [ ]:
data.shape


### Variabilidad en los predictores


In [ ]:
#data=data.drop(['1/pbi_pc'],axis=1)
data = pd.read_csv('Tabla 6_4.csv')

In [ ]:
data.std()

### Término de error E(ee) = 0


In [ ]:
residuos

In [ ]:
round(residuos.mean(),5)

### Multicolinealidad

#### Matriz de correlación

In [ ]:
data.corr()

In [ ]:
#fig, ax=plt.subplots(figsize=(8,6))
sns.heatmap(data.corr(), annot=True)
plt.subplots_adjust(top = 0.85)
plt.show()

#### VIF (Variance Inflation Factor)
Verificar la multicolinealidad utilizando el Factor de Inflación de la Varianza (VIF). Un VIF menor a 5 indica que no hay multicolinealidad fuerte entre las variables independientes y, por lo tanto, hay suficiente variabilidad en los predictores.







In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif

In [ ]:
X = data[['taf','pbi_pc','tft']]
exog = sm.add_constant(X)

In [ ]:
exog

In [ ]:
exog = sm.add_constant(data[data.columns[1:4]])
rango = sm.add_constant(data[data.columns[1:4]]).shape[1]

for i in range(rango):
    print(exog.columns[i] ,vif(exog.values, i))

### No sesgo de especificación

In [ ]:
y_hat = result_1.predict()

In [ ]:
y_hat

In [ ]:
fig, ax=plt.subplots(figsize=(8,6))
ax.scatter(x=y_hat , y=residuos)
ax.set_xlabel('valores estimados')
ax.set_ylabel('residuos')

plt.show()

### Homocedasticidad

In [ ]:
plt.plot(residuos)

In [ ]:
plt.scatter(y_hat , residuos**2)

#### Test Bresch-Pagan

In [ ]:
var_expl = result_1.model.exog

In [ ]:
test_hete_bp = sms.het_breuschpagan(residuos, var_expl) # H0: Homocedasticidad

In [ ]:
test_hete_bp

H0 : Homocedasticidad

In [ ]:
labels_test = ['estad_LM','p-value LM','estad_F','p-value F']


In [ ]:
pd.Series(test_hete_bp , index = labels_test)

#### Test White

In [ ]:
test_hete_w = sms.het_white(residuos, var_expl)

In [ ]:
test_hete_w

In [ ]:
pd.Series(test_hete_w , index = labels_test)

### No Autocorrelación

#### Test Durbin-Watson

In [ ]:
plt.scatter(y=residuos, x=data.index)

In [ ]:
resi = pd.DataFrame(residuos).set_index(data.index)

In [ ]:
resi.columns = ['residuos']

In [ ]:
resi['residuos-1'] = resi.residuos.shift()

In [ ]:
resi

In [ ]:
test_autocorr_dw = sms.durbin_watson(residuos)

test_autocorr_dw

#### Test Breusch - Godfrey

In [ ]:
test_autocorr_bg = sms.acorr_breusch_godfrey(result_1, nlags= 1) #H0 : No autocorrelación

In [ ]:
pd.Series(test_autocorr_bg , index = labels_test)

H0 : No autocorrelación

### Normalidad

#### Test Jarque-Bera

In [ ]:
residuos = result_1.resid

In [ ]:
test_normal_jb = sms.jarque_bera(residuos) # Ho: Normailidad

In [ ]:
test_normal_jb

In [ ]:
labels_test_jb = ['Jarque_Bera','p-value','asimetria','curtosis']

In [ ]:
pd.Series(test_normal_jb , labels_test_jb)

CA = skew o asimetría

EC = curtosis -3


In [ ]:
plt.hist(residuos, bins=20)

#### Bonus: Exportar multiples regresiones en columnas

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

#Este modulo de statsmodels nos permite exportar en columnas los resultados
from statsmodels.iolib.summary2 import summary_col

#Datos de prueba
x = [1, 3, 5, 6, 8, 3, 4, 5, 1, 3, 5, 6, 8, 3, 4, 5, 0, 1, 0, 1, 1, 4, 5, 7]
y = [0, 1, 0, 1, 1, 4, 5, 7,0, 1, 0, 1, 1, 4, 5, 7,0, 1, 0, 1, 1, 4, 5, 7]
d = { "x": pd.Series(x), "y": pd.Series(y)}
df = pd.DataFrame(d)
df['xsqr'] = df['x']**2

#Regresiones
mod = smf.ols('y ~ x', data=df)
res = mod.fit()
print(res.summary())
df['xcube'] = df['x']**3

mod2= smf.ols('y ~ x + xsqr', data=df)
res2 = mod2.fit()
print(res2.summary())

mod3= smf.ols('y ~ x + xsqr + xcube', data=df)
res3 = mod3.fit()
print(res3.summary())

dfres = summary_col([res,res2,res3],stars=True)
print(dfres)